# NLP Assignment 2

**Student Name:** Abubakar Imran  
**Roll Number:** i23-2589

---

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import collections
import os
import json
import math
import random

## 1. Data Loading and Preprocessing

In [30]:
def load_and_tokenize(file_path):
    if not os.path.exists(file_path):
        return []
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    return text.split()

def build_vocabulary(tokens, max_vocab_size=10000):
    word_counts = collections.Counter(tokens)
    common_words = word_counts.most_common(max_vocab_size - 1)
    word2idx = {"<UNK>": 0}
    idx2word = {0: "<UNK>"}
    for word, _ in common_words:
        if word not in word2idx:
            idx = len(word2idx)
            word2idx[word] = idx
            idx2word[idx] = word
    return word2idx, idx2word, word_counts

def preprocess_corpus(tokens, word2idx):
    return [word2idx.get(token, word2idx["<UNK>"]) for token in tokens]

In [31]:
DATA_PATH = "data/cleaned.txt"
tokens = load_and_tokenize(DATA_PATH)
word2idx, idx2word, word_counts = build_vocabulary(tokens, max_vocab_size=10000)
tokenized_corpus = preprocess_corpus(tokens, word2idx)
print(f"Loaded {len(tokens)} tokens. Vocab size: {len(word2idx)}")

Loaded 316906 tokens. Vocab size: 10000


## 2. Word2Vec Evaluation (Urdu Language)

In [32]:
w2v_embeddings = np.load("embeddings/embeddings_w2v.npy")

def calculate_mrr(test_set, embeddings, word2idx, idx2word):
    rec_ranks = []
    
    for (a, b, c), target in test_set:
        if not all(w in word2idx for w in [a, b, c, target]):
            continue
            
        target_vec = embeddings[word2idx[b]] - embeddings[word2idx[a]] + embeddings[word2idx[c]]
        norms = np.linalg.norm(embeddings, axis=1)
        t_norm = np.linalg.norm(target_vec)
        sims = np.dot(embeddings, target_vec) / (norms * t_norm + 1e-9)
        
        for w in [a, b, c]: sims[word2idx[w]] = -1
            
        rank = np.sum(sims > sims[word2idx[target]]) + 1
        rec_ranks.append(1 / rank)
    
    return np.mean(rec_ranks) if rec_ranks else 0

In [33]:
# Analogy tasks specifically for Urdu grammar found in your corpus
# (a : b :: c : ?)
urdu_eval_set = [
    (("تھا", "تھے", "ہے"), "ہیں"),  # Singular/Plural Tense (was:were :: is:are)
    (("کا", "کی", "کے"), "میں"),   # Possessive/Gender markers
    (("ان", "اس", "وہ"), "انھوں")  # Pronoun variants
]

mrr_score = calculate_mrr(urdu_eval_set, w2v_embeddings, word2idx, idx2word)
print(f"Urdu MRR Score: {mrr_score:.4f}")

print("\nSample Nearest Neighbors (Urdu):")
for word in ["پاکستان", "پاکستان", "ہے", "ان"]:
    # Note: re-using the logic from previous cell for quick check
    if word in word2idx:
        target_vec = w2v_embeddings[word2idx[word]]
        sims = np.dot(w2v_embeddings, target_vec) / (np.linalg.norm(w2v_embeddings, axis=1) * np.linalg.norm(target_vec) + 1e-9)
        top_idx = np.argsort(sims)[-6:-1][::-1]
        neighbors = [idx2word[i] for i in top_idx]
        print(f"{word}: {', '.join(neighbors)}")

Urdu MRR Score: 0.1668

Sample Nearest Neighbors (Urdu):
پاکستان: تجارت, ’باہمی, ادارہ, پاکستانی, انڈین
پاکستان: تجارت, ’باہمی, ادارہ, پاکستانی, انڈین
ہے: ہے‘, ہے،, سیٹلائیٹ, اگرچہ, تھا
ان: بتانے, 'تمام, انسانی, تعاقب،, خاندانوں
